# Week 8

### Goals  
- Learn simple retrieval without a vector database.
- Add trustworthy text fields to a small, manageable subset.

### Tasks  
- Prepare a small licensed or self-authored description file; optionally enrich selected titles via TMDB if a free
account and current terms permit it.
- Document provenance and required attribution for any external metadata.
- Vectorize descriptions or tags with scikit-learn TF-IDF.
- Rank candidates by cosine similarity to a user request.
- Compare retrieval with genre filtering and inspect at least five results.
- If descriptions are unavailable, use MovieLens tags and title/genre text as the retrieval corpus.

### Expected Output  
- 08_retrieval.ipynb and a documented text corpus.
- A search_descriptions() tool and qualitative retrieval examples.

---
#### Setup

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append("../")
from src.catalog import catalog
from src.tools import filter_movies, search_descriptions
from src.retrieval import CORPUS_PATH, MAX_RESULTS, build_corpus, _get_index

RAW_TAGS_PATH = Path("../data/ml-latest-small/tags.csv")    # original MovieLens download

#### Quick overview of the tags CSV

In [3]:
tags = pd.read_csv(RAW_TAGS_PATH)   
print(len(tags), "tag rows,", tags["movieId"].nunique(), "movies with tags")
print(tags["tag"].str.lower().value_counts().head(15))
print(tags.groupby("movieId").size().describe())

3683 tag rows, 1572 movies with tags
tag
in netflix queue      131
atmospheric            41
funny                  24
superhero              24
surreal                24
thought-provoking      24
sci-fi                 23
disney                 23
quirky                 22
religion               22
dark comedy            21
suspense               21
psychology             21
twist ending           20
visually appealing     20
Name: count, dtype: int64
count    1572.000000
mean        2.342875
std         5.562342
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max       181.000000
dtype: float64


#### Build the Corpus

For this Notebook I chose to work with the `tags.csv` data already downloaded in week 3 without enriching it with data from TMDB. The main reason for that is that I first want to focus on building a manageable corpus and understand how it all works. If time allows it I will expand the corpus with data from TMDB for the final version.

**Corpus construction**: tags lowercased, one entry per unique tag per movie, the tag "in netflix queue" removed (as seen in print above occurs 131 times, but this tag does not add any relevant information to the movie), hyphens removed for matching, movies with fewer than 3 ratings excluded.

In [4]:
corpus = build_corpus()
print(f"Corpus: {len(corpus)} of {len(catalog)} catalog movies ({len(corpus) / len(catalog):.1%}) have tags")
print(corpus["tags"].map(len).describe())

# Provenance file: what the corpus contains (documentation only, nothing reads it back)
corpus_out = corpus[["movieId", "title", "year", "genres", "tags"]].copy()
corpus_out["tags"] = corpus_out["tags"].map("; ".join)
corpus_out.to_csv(CORPUS_PATH, index=False)

Corpus: 1214 of 9724 catalog movies (12.5%) have tags
count    1214.000000
mean        2.546952
std         5.911328
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max       173.000000
Name: tags, dtype: float64


There seems to be a movie with 173 tags. That is quite a lot and much more than most other movies. Ich quickly check that here to make sure this is not a mistake.

In [8]:
print(corpus.loc[corpus["tags"].map(len).idxmax(), "title"], corpus["tags"].map(len).max())
print(corpus[corpus["title"].str.contains("Fight Club|Usual Suspects")][["title", "tags"]])

Pulp Fiction 173
                  title                                               tags
26   The Usual Suspects  [heist, mindfuck, suspense, thriller, tricky, ...
579          Fight Club  [action, atmospheric, based on a book, brad pi...


#### Check TF-IDF index

In [5]:
_, vectorizer, matrix = _get_index()
print("index shape (movies x terms):", matrix.shape)

# idf = how rare a term is across movies. Higher idf = rarer = more weight in the similarity.
idf = dict(zip(vectorizer.get_feature_names_out(), vectorizer.idf_))
for term in ["superhero", "atmospheric", "scifi", "comedy", "drama"]:
    print(f"{term:12s} idf = {idf.get(term, float('nan')):.2f}")

index shape (movies x terms): (1214, 8395)
superhero    idf = 4.97
atmospheric  idf = 4.52
scifi        idf = 3.09
comedy       idf = 2.03
drama        idf = 1.62


**Direct tests of the search function**

In [9]:
try:
    search_descriptions("   ")
except ValueError as e:
    print("empty query rejected:", e)

assert search_descriptions("asffbaisfb") == [], "nonsense must return an empty list"

res = search_descriptions("twist ending")
assert all(r["similarity"] > 0 for r in res)
assert all(r["movieId"] in set(catalog["movieId"]) for r in res)
assert len(res) <= MAX_RESULTS
print("direct tests passed")
display(pd.DataFrame(res))

empty query rejected: query must not be empty.
direct tests passed


,movieId,title,year,genres,mean,count,similarity,matched_tags,matched_genres
0,2529,Planet of the Apes,1968,"[Action, Drama, Sci-Fi]",3.804,56,0.437,[twist ending],[]
1,1834,The Spanish Prisoner,1997,"[Crime, Drama, Mystery, Thriller]",4.000,11,0.429,[twist ending],[]
2,47,Seven,1995,"[Mystery, Thriller]",3.975,203,0.396,[twist ending],[]
3,1625,The Game,1997,"[Drama, Mystery, Thriller]",3.701,77,0.392,"[plot twist, twist ending]",[]
4,44665,Lucky Number Slevin,2006,"[Crime, Drama, Mystery]",3.855,38,0.389,[twist ending],[]


**Comparison with genre filtering**

In [ ]:
# Compare search results with genre-based filtering
comparisons = [
    ("twist ending", "Thriller"),
    ("atmospheric sci-fi", "Sci-Fi"),
    ("surreal dark comedy", "Comedy"),
    ("thought-provoking psychology", "Drama"),
    ("superhero", "Action"),
    ("feel-good movie for a rainy sunday", "Comedy"),
]

for query, genre in comparisons:
    retrieved = search_descriptions(query)
    by_genre = filter_movies(genre=genre)
    print(f"\nQUERY: {query!r}")
    print(f"  retrieval ({len(retrieved)} results):")
    for r in retrieved:
        print(f"    {r['title']} ({r['year']}) sim={r['similarity']} tags={r['matched_tags']} genres={r['matched_genres']}")
    print(f"  genre filter {genre!r} (top {len(by_genre)} by score): "
          + ", ".join(m["title"] for m in by_genre[:5]) + " ...")


QUERY: 'twist ending'
  retrieval (5 results):
    Planet of the Apes (1968) sim=0.437 tags=['twist ending'] genres=[]
    The Spanish Prisoner (1997) sim=0.429 tags=['twist ending'] genres=[]
    Seven (1995) sim=0.396 tags=['twist ending'] genres=[]
    The Game (1997) sim=0.392 tags=['plot twist', 'twist ending'] genres=[]
    Lucky Number Slevin (2006) sim=0.389 tags=['twist ending'] genres=[]
  genre filter 'Thriller' (top 10 by score): Fight Club, The Usual Suspects, The Departed, Rear Window, Pulp Fiction ...

QUERY: 'atmospheric sci-fi'
  retrieval (5 results):
    The Prestige (2006) sim=0.254 tags=['atmospheric'] genres=['Sci-Fi']
    Sicario (2015) sim=0.204 tags=['atmospheric'] genres=[]
    Upstream Color (2013) sim=0.194 tags=['atmospheric'] genres=['Sci-Fi']
    Watership Down (1978) sim=0.19 tags=['atmospheric'] genres=[]
    Lost in Translation (2003) sim=0.183 tags=['atmospheric'] genres=[]
  genre filter 'Sci-Fi' (top 10 by score): Star Wars: Episode IV - A New Hope

#### Observations

**What was built:** a tag-based retrieval tool, `search_descriptions(query)` (`src/retrieval.py`): TF-IDF over title, genres and unique user tags, cosine similarity ranking, at most 5 results, never padded with zero-similarity movies. Each result carries `matched_tags` and `matched_genres` as citeable evidence.

**What worked**
- *Theme queries that genre cannot express.* "superhero" returned Superman, Batman, Supergirl, Spider-Man and Superman II. "twist ending" returned Planet of the Apes, Seven and The Game, each with the matching tag shown.
- *Genre filtering ignores mood.* Its top-10 list is the best-rated films of the genre, so Fight Club appears in three of six lists, and "surreal dark comedy" and "feel-good movie for a rainy sunday" received the identical Comedy list. Retrieval reacts to the query.
- *Filler words matter.* Before adding "movie/film" as stop words, the rainy-Sunday query returned films matched only on the word "movie" (tags such as "movie business"). After the fix: 3 results, all with the tag "feel-good".

**Limitations (with evidence)**
- *Coverage:* only 12.5% of the catalog has tags, and the median movie has 1 tag, so many relevant films cannot be found at all.
- *Source bias:* three users wrote about 63% of all raw tag rows, so the corpus mostly reflects their vocabulary. Tags are subjective keywords, not descriptions: a match shows someone labeled the movie that way, not that the movie has the quality.
- *Vocabulary mismatch:* matching works on words, not meaning. The Usual Suspects is tagged ("mindfuck", "tricky") but does not match "twist ending".
- *No AND logic:** for "atmospheric sci-fi" the rarer word dominates (IDF 4.52 for "atmospheric" vs 3.09 for "scifi"), so only 2 of 5 results are Sci-Fi. For "surreal dark comedy" 4 of 5 are dark comedies and none are surreal.
- *Very long documents:* Pulp Fiction has 173 unique tags, which dilutes each term's weight.
- *Similarity values (0.2 to 0.4) are rank scores, not confidence.*

**Consequences for the agent**
- The tool is tested on its own but not yet in the agent's tool list.
- The no-AND-logic problem suggests a pipeline: `filter_movies` narrows by genre or year first, then `search_descriptions` ranks the candidates.
- `matched_tags` gives the agent real evidence to cite, which addresses the unsupported-claims failure mode seen in Week 7.
- Retrieval is only as trustworthy as the tags: reasons built from it should say "tagged as ..." and not "is ...".